Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 04 — eksperymentalne porównanie strategii optymalizacji

Materiał towarzyszący [`README.pl.md`](README.pl.md). Regresja na danych o
celowo różnych skalach cech — porównujesz SGD, momentum i Adam, badasz
wpływ learning rate, diagnozujesz nieudany trening, i oceniasz uczciwie
na zbiorze testowym.




In [ ]:
import matplotlib.pyplot as plt  # noqa: F401 - uzywany w sekcjach 3/4/6
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")


## 1. Dane (gotowe)

Trzy cechy o różnych skalach: $x_1\in[-1,1]$, $x_2\in[-40,40]$,
$x_3\in[-3,3]$; $y = 3\sin(1{,}5 x_1) + 0{,}05 x_2 + 0{,}8 x_3 + \varepsilon$.
Podział 70/15/15 (trening/walidacja/test).


In [ ]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    x1 = torch.empty(n).uniform_(-1, 1, generator=g)
    x2 = torch.empty(n).uniform_(-40, 40, generator=g)
    x3 = torch.empty(n).uniform_(-3, 3, generator=g)
    noise = torch.randn(n, generator=g) * 0.5
    y = 3.0 * torch.sin(1.5 * x1) + 0.05 * x2 + 0.8 * x3 + noise
    X = torch.stack([x1, x2, x3], dim=1)
    return X, y.unsqueeze(-1)


X, y = make_data(1600, seed=0)
n = X.shape[0]
perm = torch.randperm(n, generator=torch.Generator().manual_seed(1))
n_val = int(0.15 * n)
n_test = int(0.15 * n)
val_idx = perm[:n_val]
test_idx = perm[n_val : n_val + n_test]
train_idx = perm[n_val + n_test :]

X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_val, y_val = X[val_idx].to(device), y[val_idx].to(device)
X_test, y_test = X[test_idx].to(device), y[test_idx].to(device)

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}, N_test={len(test_idx)}")
print(f"odchylenia std cech: {X_train.std(dim=0).tolist()}")


## 2. Pętla treningowa — Twoje zadanie


In [ ]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)


def iterate_minibatches(X, y, batch_size, seed):
    """Jawne tworzenie mini-batchy: permutacja indeksow, pociecie na kawalki."""
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0]
    perm = torch.randperm(n, generator=g)
    for i in range(0, n, batch_size):
        idx = perm[i : i + batch_size]
        yield X[idx], y[idx]


In [ ]:
def train_model(opt_name, lr, epochs, batch_size=32, model_seed=42, data_seed=123, momentum=0.9):
    """Trenuje model (opt_name: 'sgd'|'momentum'|'adam'), zwraca
    (model, train_losses, val_losses) — strata MSE po kazdej epoce.
    """
    # TODO:
    # 1. model = make_model(model_seed)
    # 2. zbuduj optymalizator: torch.optim.SGD(..., lr=lr) albo
    #    torch.optim.SGD(..., lr=lr, momentum=momentum) albo torch.optim.Adam(..., lr=lr)
    #    w zaleznosci od opt_name
    # 3. loss_fn = nn.MSELoss()
    # 4. w petli po epochs: dla kazdego mini-batcha z iterate_minibatches
    #    (uzyj seed=data_seed+epoch, zeby kolejnosc batchy zmieniala sie
    #    miedzy epokami, ale byla powtarzalna) wykonaj:
    #    zero_grad() -> forward -> strata -> backward() -> step()
    # 5. po kazdej epoce policz i zapisz strate na X_train/y_train i X_val/y_val
    #    (pod torch.no_grad())
    raise NotImplementedError("TODO: train_model")


In [ ]:
baseline_val = ((y_val - y_train.mean()) ** 2).mean().item()
baseline_test = ((y_test - y_train.mean()) ** 2).mean().item()
print(f"Baseline (srednia z treningu): val MSE={baseline_val:.4f}  test MSE={baseline_test:.4f}")


## 3. Porównanie przy wspólnym learning rate — Twoje zadanie


In [ ]:
# TODO: wywolaj train_model dla "sgd", "momentum", "adam" z lr=0.01, epochs=20.
# Zbierz wyniki (np. w slowniku common_results[name] = (tl, vl)) i narysuj
# krzywe straty treningowej i walidacyjnej dla wszystkich trzech (2 wykresy
# obok siebie albo jeden z 3 liniami kazdy).
raise NotImplementedError("TODO: porownanie przy wspolnym lr")


## 4. Przegląd learning rate — Twoje zadanie


In [ ]:
# TODO: dla SGD przetestuj lr z [0.0001, 0.001, 0.01] (epochs=20 kazdy),
# wypisz strate walidacyjna koncowa dla kazdego. Zrob to samo dla momentum
# (momentum=0.9) z lr z [0.00005, 0.0005, 0.005].
raise NotImplementedError("TODO: przeglad learning rate")


## 5. Diagnoza — Twoje zadanie

**Zadanie (Markdown, 4–6 zdań):** w sekcji 3 przynajmniej jeden
optymalizator (poza Adamem) przy `lr=0,01` prawdopodobnie nie nauczył się
niczego sensownego (strata mogła wybuchnąć do `nan`). Na podstawie
wyników z sekcji 4 i tego, co wiesz z wykładu o różnych skalach cech i
krzywiźnie straty: zdiagnozuj **dlaczego** akurat `lr=0,01` zawiódł dla
tego optymalizatora, a dla Adama nie. Nie wystarczy "lr było za duże" —
wyjaśnij, co w *tych konkretnych danych* sprawia, że SGD/momentum są tak
wrażliwe na dobór `lr`.


_(Twoja odpowiedź tutaj.)_


## 6. Finalna ewaluacja — Twoje zadanie

Wybierz **jeden, najlepszy wg straty walidacyjnej** `lr` dla każdego z
trzech optymalizatorów (na podstawie sekcji 4 — dla Adama możesz przyjąć
`lr=0,01` z sekcji 3). Wytrenuj każdy z wybranym `lr` (`epochs=25`), a
następnie oceń **raz** na zbiorze testowym.


In [ ]:
# TODO:
# 1. zdefiniuj best_lrs = {"sgd": ..., "momentum": ..., "adam": 0.01} na
#    podstawie Twoich wynikow z sekcji 4
# 2. dla kazdego optymalizatora: train_model(..., lr=best_lrs[name], epochs=25)
# 3. policz strate na X_test/y_test (pod torch.no_grad(), nn.MSELoss())
# 4. wypisz tabele: optymalizator, lr, strata treningowa, walidacyjna, TESTOWA
raise NotImplementedError("TODO: finalna ewaluacja")


## 7. Wnioski — Twoje zadanie

Odpowiedz (Markdown, uzasadnione realnymi liczbami z sekcji 6):

1. Który optymalizator osiągnął najniższą stratę testową? Czy ten wynik
   pasuje do tego, czego nauczył Cię wykład o problemach źle
   uwarunkowanych przez różne skale cech?
2. Gdybyś porównał/a te same trzy optymalizatory na problemie **bez**
   różnicy skal cech (wszystkie cechy znormalizowane) — czy spodziewasz
   się podobnie dużej różnicy między nimi? Na czym opierasz tę
   przewidywanie?


_(Twoja odpowiedź tutaj.)_
